In [1]:
#clear all
%reset -f

#import packages
import numpy as np
import sys
import os
import pandas as pd
import mne
import matplotlib
from sklearn.utils import resample
from mne_icalabel import label_components

#specify working directory
root = 'F:/Documents/Science/MirRevAdaptEEG'
participants = list(range(0,32))
#specify which erp we are analyzing
# this controls which event we time-lock analysis to
# 'frn' will be time-locked to feedback onset, 'ern' is time-locked to movement onset, 'lrp' time-locked to go signal onset
erps = 'frn'

#pop up plots as separate window & interactive
%matplotlib qt
matplotlib.pyplot.close('all')

In [2]:
#setting up path/ directory
#access specific file for specific participant
def load_epochs(pp_num, root_dir, erp_path):

    root_directory = root_dir
    data_directory = os.path.join(root_directory, 'data/eeg/')
    id_directory = os.path.join(data_directory, 'p%03d/' % pp_num)
    pp_directory = os.path.join(id_directory, erp_path)
    filename = os.path.join(pp_directory, 'mra_p%03d_run0-epo.fif' % pp_num)

    epochs = mne.read_epochs(filename)
    #baseline for each trial
    if erp_path == 'frn':
        epochs = epochs.apply_baseline(baseline = (-0.1, 0))
    elif erp_path == 'lrp':
        epochs = epochs.apply_baseline(baseline = (-1.3, -1.0))
    elif erp_path == 'ern':
        epochs = epochs.apply_baseline(baseline = (-1.35, -1.05))

    return epochs, pp_directory

In [3]:
#get index of trials from behavioral data
#rdm before rot and rdm before mir are read in as two different behavioral data
def epoch_index(task, listval, pp, target = np.nan):
    if task == 'rot':
        df = pd.read_csv(r'F:\Documents\Science\MirRevAdaptEEG\data\ROT_learningcurve_degrees.csv')
    elif task == 'rdmrot':
        df = pd.read_csv(r'F:\Documents\Science\MirRevAdaptEEG\data\RDMROT_learningcurve_degrees.csv')
    elif task == 'rdmmir':
        df = pd.read_csv(r'F:\Documents\Science\MirRevAdaptEEG\data\RDMMIR_learningcurve_degrees.csv')
    elif task == 'aln':
        df = pd.read_csv(r'F:\Documents\Science\MirRevAdaptEEG\data\ALIGNED_learningcurve_degrees.csv')
    elif task == 'mir':
        df = pd.read_csv(r'F:\Documents\Science\MirRevAdaptEEG\data\MIR_learningcurve_degrees.csv')
   
    ppdata = list(df.iloc[:,pp+1])
    
    index = [i for i,v in enumerate(ppdata) if listval[0] <= i < listval[1]]
    
    #epochs_out = epochs[index]
    numtrials = len(index)
    
    #return epochs_out, numtrials
    return index, numtrials

In [4]:
#separate function to get indexed epochs for random, since we have them before rotation or mirror
def trial_list_rdm(task, pp, index):
    df = pd.read_csv(r'F:\Documents\Science\MirRevAdaptEEG\data\eeg\p%03d\%s\mra_p%03d_tasktrigindex.csv' % (pp, erps, pp))
    
    #even pp numbers would experience rotation first (need random before this)
    #need to also take into account missing triggers from previous tasks
    if pp % 2 == 0:
        if task == 'rdmrot':
            #get trial after first task (trial 48 to 95, 234 to 281)
            tasksdf = df[0:96]
            subtasksdf = tasksdf.loc[tasksdf['tasktrig'] == 16150]
            triallist = []
            for i in index:
                ndat = subtasksdf.loc[subtasksdf['trialno'] == i]
                nint = int(ndat['trigidx'])
                if nint == 1:
                    trial = int(ndat['ctrial'])
                    triallist.append(trial)
            
                    
        elif task == 'rdmmir':
            #get trial after first task (trial 48 to 95)
            tasksdf = df[0:282]
            subtasksdf = tasksdf.loc[tasksdf['tasktrig'] == 16153]
            triallist = []
            for i in index:
                ndat = subtasksdf.loc[subtasksdf['trialno'] == i]
                nint = int(ndat['trigidx'])
                if nint == 1:
                    trial = int(ndat['ctrial'])
                    triallist.append(trial)
            
            
    elif pp % 2 == 1:   
        if task == 'rdmrot':
            #get trial after first five tasks (trial 234 to 281)
            tasksdf = df[0:282]
            subtasksdf = tasksdf.loc[tasksdf['tasktrig'] == 16153]
            triallist = []
            for i in index:
                ndat = subtasksdf.loc[subtasksdf['trialno'] == i]
                nint = int(ndat['trigidx'])
                if nint == 1:
                    trial = int(ndat['ctrial'])
                    triallist.append(trial)
            
            
        elif task == 'rdmmir':
            #get trial after first task (trial 48 to 95)
            tasksdf = df[0:96]
            subtasksdf = tasksdf.loc[tasksdf['tasktrig'] == 16150]
            triallist = []
            for i in index:
                ndat = subtasksdf.loc[subtasksdf['trialno'] == i]
                nint = int(ndat['trigidx'])
                if nint == 1:
                    trial = int(ndat['ctrial'])
                    triallist.append(trial)
            

    
    return(triallist)
    

In [5]:
#get matching trials from eeg data, where we account for missing triggers
def epoch_trials(task, pp, epochs, index):

    df = pd.read_csv(r'F:\Documents\Science\MirRevAdaptEEG\data\eeg\p%03d\%s\mra_p%03d_tasktrigindex.csv' % (pp, erps, pp))
    
    if task == 'aln':
        #get trial numbers where tasktrig is 16149 and trigidx is 1
        tasksdf = df[0:48]
        subtasksdf = tasksdf.loc[tasksdf['tasktrig'] == 16149]
        triallist = []
        for i in index:
            ndat = subtasksdf.loc[subtasksdf['trialno'] == i]
            nint = int(ndat['trigidx'])
            if nint == 1:
                trial = int(ndat['ctrial'])
                triallist.append(trial)

        epochs = epochs[triallist]
    
    #for the other tasks, the order depends on their pp number
    #even pp numbers would experience rotation first (need random before this)
    #need to also take into account missing triggers from previous tasks
    if pp % 2 == 0:
        if task == 'rot':
            #rot trials are after first two tasks (trial 96 to 185)
            tasksdf = df[0:186]
            subtasksdf = tasksdf.loc[tasksdf['tasktrig'] == 16151]
            triallist = []
            for i in index:
                ndat = subtasksdf.loc[subtasksdf['trialno'] == i]
                nint = int(ndat['trigidx'])
                if nint == 1:
                    trial = int(ndat['ctrial'])
                    triallist.append(trial)

            epochs = epochs[triallist]
            
        elif task == 'mir':
            #get trial after first task (trial 282 to 371)
            tasksdf = df[0:372]
            subtasksdf = tasksdf.loc[tasksdf['tasktrig'] == 16154]
            triallist = []
            for i in index:
                ndat = subtasksdf.loc[subtasksdf['trialno'] == i]
                nint = int(ndat['trigidx'])
                if nint == 1:
                    trial = int(ndat['ctrial'])
                    triallist.append(trial)

            epochs = epochs[triallist]
            
    elif pp % 2 == 1:
        if task == 'rot':
            #get trial after first five tasks (trial 282 to 371)
            tasksdf = df[0:372]
            subtasksdf = tasksdf.loc[tasksdf['tasktrig'] == 16151]
            triallist = []
            for i in index:
                ndat = subtasksdf.loc[subtasksdf['trialno'] == i]
                nint = int(ndat['trigidx'])
                if nint == 1:
                    trial = int(ndat['ctrial'])
                    triallist.append(trial)

            epochs = epochs[triallist]
            
        elif task == 'mir':
            #get trial after first five tasks (trial 96 to 185)
            tasksdf = df[0:186]
            subtasksdf = tasksdf.loc[tasksdf['tasktrig'] == 16154]
            triallist = []
            for i in index:
                ndat = subtasksdf.loc[subtasksdf['trialno'] == i]
                nint = int(ndat['trigidx'])
                if nint == 1:
                    trial = int(ndat['ctrial'])
                    triallist.append(trial)

            epochs = epochs[triallist]
    
    return(epochs)

In [6]:
#save epochs
def output_condition_epochs(pp_num, data, ppdir, condname, task):
    # Save the cleaned data :)
    out_fname = os.path.join(ppdir, 'p%03d_%s_%s-epo.fif' % (pp_num, condname, task))
    data.save(out_fname, overwrite = True)

In [7]:
#save as an evoked object
def output_condition_evoked(pp_num, data, ppdir, condname, task):
    evoked = data.average()
    out_fname = os.path.join(ppdir, 'p%03d_%s_%s-ave.fif' % (pp_num, condname, task))
    evoked.save(out_fname, overwrite = True)

Please note that the cells below should be executed depending on what output you prefer (i.e., which erp you are time-locked to).

In [8]:
#save as evoked objects - runs for ANY erp type
#each perturbation condition can have separate loops

#lowpass filter cutoff
l_cutoff = 30

#ALIGNED
#have one condition for aligned
conditions = [0,48]

for pp in participants:
    data, ppdir = load_epochs(pp, root_dir = root, erp_path = erps)
    idx, numtrials = epoch_index('aln', conditions, pp)
    #print(idx)
    aldata = epoch_trials('aln', pp, data, idx)
    #print(aldata)
    output_condition_epochs(pp, aldata, ppdir, 'early_late', 'aligned')
    #filter at 30 Hz
    aldata = aldata.filter(l_freq= None, h_freq=l_cutoff)
    #then average across trials
    output_condition_evoked(pp, aldata, ppdir, 'early_late', 'aligned')
        
#RANDOM 
#combine rdm rot and rdm mir into one per participant, but get correct indices
# early1 is now defined by the first rdm block they do, late1 will be the last blocks they do in the first rdm block
# early2 is now defined by the second rdm block they do, late2 will be the last blocks they do in the second rdm block
conditions = [[0,12], [24,48], [0,12], [24,48]] #1st two blocks, last 4 blocks of every rdm block
conditionnames = ['early1', 'late2', 'early2', 'late1']

for pp in participants:
    data, ppdir = load_epochs(pp, root_dir = root, erp_path = erps)
    for cond in range(0, len(conditions)):
        if pp % 2 == 0 and cond == 0:
            idx, numtrials = epoch_index('rdmrot', conditions[cond], pp)
            rdm_early1_trials = trial_list_rdm('rdmrot', pp, idx)
            pdata = data[rdm_early1_trials]
        elif pp % 2 == 0 and cond == 1:
            idx, numtrials = epoch_index('rdmmir', conditions[cond], pp)
            rdm_late2_trials = trial_list_rdm('rdmmir', pp, idx)
            pdata = data[rdm_late2_trials]
        elif pp % 2 == 1 and cond == 0:
            idx, numtrials = epoch_index('rdmmir', conditions[cond], pp)
            rdm_early1_trials = trial_list_rdm('rdmmir', pp, idx)
            pdata = data[rdm_early1_trials]
        elif pp % 2 == 1 and cond == 1:
            idx, numtrials = epoch_index('rdmrot', conditions[cond], pp)
            rdm_late2_trials = trial_list_rdm('rdmrot', pp, idx)
            pdata = data[rdm_late2_trials]
        elif pp % 2 == 0 and cond == 2:
            idx, numtrials = epoch_index('rdmmir', conditions[cond], pp)
            rdm_early2_trials = trial_list_rdm('rdmmir', pp, idx)
            pdata = data[rdm_early2_trials]
        elif pp % 2 == 0 and cond == 3:
            idx, numtrials = epoch_index('rdmrot', conditions[cond], pp)
            rdm_late1_trials = trial_list_rdm('rdmrot', pp, idx)
            pdata = data[rdm_late1_trials]
        elif pp % 2 == 1 and cond == 2:
            idx, numtrials = epoch_index('rdmrot', conditions[cond], pp)
            rdm_early2_trials = trial_list_rdm('rdmrot', pp, idx)
            pdata = data[rdm_early2_trials]
        elif pp % 2 == 1 and cond == 3:
            idx, numtrials = epoch_index('rdmmir', conditions[cond], pp)
            rdm_late1_trials = trial_list_rdm('rdmmir', pp, idx)
            pdata = data[rdm_late1_trials]
            
        output_condition_epochs(pp, pdata, ppdir, conditionnames[cond], 'rdm')
        #filter at 30 Hz
        pdata = pdata.filter(l_freq= None, h_freq=l_cutoff)
        #then average across trials
        output_condition_evoked(pp, pdata, ppdir, conditionnames[cond], 'rdm')

Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p000\frn\mra_p000_run0-epo.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms
        0 CTF compensation matrices available
Not setting metadata
420 matching events found
No baseline correction applied
0 projection items activated
Applying baseline correction (mode: mean)
Overwriting existing file.
Setting up low-pass filter at 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 89 samples (0.445 sec)



[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   4 out of   4 | elapsed:    0.0s remaining:    0.0s


Overwriting existing file.
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p001\frn\mra_p001_run0-epo.fif ...


[Parallel(n_jobs=1)]: Done 3072 out of 3072 | elapsed:    1.0s finished


    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms
        0 CTF compensation matrices available
Not setting metadata
420 matching events found
No baseline correction applied
0 projection items activated
Applying baseline correction (mode: mean)
Overwriting existing file.
Setting up low-pass filter at 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 89 samples (0.445 sec)



...

[Parallel(n_jobs=1)]: Done 1536 out of 1536 | elapsed:    0.4s finished
[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   4 out of   4 | elapsed:    0.0s remaining:    0.0s


Overwriting existing file.
Overwriting existing file.
Setting up low-pass filter at 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 89 samples (0.445 sec)



[Parallel(n_jobs=1)]: Done 768 out of 768 | elapsed:    0.2s finished
[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done   4 out of   4 | elapsed:    0.0s remaining:    0.0s


Overwriting existing file.


[Parallel(n_jobs=1)]: Done 1536 out of 1536 | elapsed:    0.4s finished


In [9]:
#get grand averages - split according to early or late conditions; - runs for ANY erp type
root_directory = root
data_directory = os.path.join(root_directory, 'data/eeg/')

#read in evoked object
conditionnames = ['early1', 'late2', 'early2', 'late1']

grand_aligned = []
grand_rdm_early1 = []
grand_rdm_late2 = []
grand_rdm_early2 = []
grand_rdm_late1 = []

#aligned
flist = []
evoked_list = []

for pp in participants:
    id_directory = os.path.join(data_directory, 'p%03d/' % participants[pp])
    pp_directory = os.path.join(id_directory, erps)
    fname = os.path.join(pp_directory, 'p%03d_%s_%s-ave.fif' % (participants[pp], 'early_late', 'aligned'))
    evoked = mne.read_evokeds(fname)
    flist.append(evoked)
    evoked = flist[pp][0]
    evoked_list.append(evoked)
        
grand_aligned = mne.grand_average(evoked_list) 
aligned_flist = evoked_list
  
#rdm

for condition in range(0, len(conditionnames)):
    flist = []
    evoked_list = []
    for pp in participants:
        id_directory = os.path.join(data_directory, 'p%03d/' % participants[pp])
        pp_directory = os.path.join(id_directory, erps)
        fname = os.path.join(pp_directory, 'p%03d_%s_%s-ave.fif' % (participants[pp], conditionnames[condition], 'rdm'))
        evoked = mne.read_evokeds(fname)
        flist.append(evoked)
        evoked = flist[pp][0]
        evoked_list.append(evoked)
    if condition == 0:
        grand_rdm_early1 = mne.grand_average(evoked_list)
        early1_rdm_flist = evoked_list
    elif condition == 1:
        grand_rdm_late2 = mne.grand_average(evoked_list)
        late2_rdm_flist = evoked_list
    elif condition == 2:
        grand_rdm_early2 = mne.grand_average(evoked_list)
        early2_rdm_flist = evoked_list
    elif condition == 3:
        grand_rdm_late1 = mne.grand_average(evoked_list)
        late1_rdm_flist = evoked_list
            

Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p000\frn\p000_early_late_aligned-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 48 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p001\frn\p001_early_late_aligned-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 48 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p002\frn\p002_early_late_aligned-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16

Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p020\frn\p020_early_late_aligned-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 48 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p021\frn\p021_early_late_aligned-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 48 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p022\frn\p022_early_late_aligned-ave.fif ...
    Found

Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p008\frn\p008_early1_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 12 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p009\frn\p009_early1_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 12 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p010\frn\p010_early1_rdm-ave.fif ...
    Found the data of interest:
 

    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 12 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p029\frn\p029_early1_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 12 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p030\frn\p030_early1_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 12 - aspect type = 100
No projector specified fo

        0 CTF compensation matrices available
        nave = 24 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p017\frn\p017_late2_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 24 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p018\frn\p018_late2_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 24 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is bas

No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p005\frn\p005_early2_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 12 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p006\frn\p006_early2_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 12 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdapt

Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p025\frn\p025_early2_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 12 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p026\frn\p026_early2_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 12 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p027\frn\p027_early2_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compe

    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 24 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p014\frn\p014_late1_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 24 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p015\frn\p015_late1_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 24 - aspect type = 100
No projector specified for 

In [12]:
#convert evokeds to data frame, so we can plot the ERPs better in R - runs for FRN/ERN ONLY

flists = ['aln', 'early1rdm', 'late2rdm', 'early2rdm', 'late1rdm']

for f in flists:
    newdf = pd.DataFrame()
    for pp in participants:
        if f == 'aln':
            df = aligned_flist[pp].to_data_frame()
        elif f == 'early1rdm':
            df = early1_rdm_flist[pp].to_data_frame()
        elif f == 'late2rdm':
            df = late2_rdm_flist[pp].to_data_frame()
        elif f == 'early2rdm':
            df = early2_rdm_flist[pp].to_data_frame()
        elif f == 'late1rdm':
            df = late1_rdm_flist[pp].to_data_frame()
             
        time = df['time'].tolist()
        df = df[['FCz', 'F3', 'Fz', 'F4', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4']]
        #df = df[['C3', 'C4']]
        ppdat = []
        for timept in range(0,800):
            subdf = df.loc[timept,:].mean()
            ppdat.append(subdf)
    
        newdf['pp%03d'% pp] = ppdat
    
    newdf['time'] = time
    newdf_filename = os.path.join('F:/Documents/Science/MirRevAdaptEEG/data/', 'Evoked_DF_%s_%s.csv' % (f, erps))
    newdf.to_csv(newdf_filename)
        

In [13]:
#plot FRN/ERN here using MNE functions
#channels = ['FCz', 'Fz']
channels = ['FCz', 'F3', 'Fz', 'F4', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4']
#channels = ['C3']
#channels = ['C4']
# channels = ['Oz','O1','O2']
evokeds = dict(aligned = aligned_flist,
               early1_rdm = early1_rdm_flist,
               late2_rdm = late2_rdm_flist, 
               early2_rdm = early2_rdm_flist, 
               late1_rdm = late1_rdm_flist)
#plot
mne.viz.plot_compare_evokeds(evokeds, picks=channels, combine = 'mean', legend = 'lower right', ylim=dict(eeg=[-5, 15]), ci=True)

More than 6 channels, truncating title ...
combining channels using "mean"
combining channels using "mean"
combining channels using "mean"
combining channels using "mean"
combining channels using "mean"


[<Figure size 800x600 with 1 Axes>]

In [14]:
#calculate difference waves for FRN/ERN
diffconds = ['early1rdm', 'late2rdm', 'early2rdm', 'late1rdm']
early1rdm_diff = []
late2rdm_diff = []
early2rdm_diff = []
late1rdm_diff = []

for cond in range(0, len(diffconds)):
    for pp in participants:
        if cond == 0:
            diffevks = mne.combine_evoked([evokeds['aligned'][pp], evokeds['early1_rdm'][pp]], 
                                                            weights=[-1, 1])
            early1rdm_diff.append(diffevks)
        elif cond == 1:
            diffevks = mne.combine_evoked([evokeds['aligned'][pp], evokeds['late2_rdm'][pp]], 
                                                         weights=[-1, 1])
            late2rdm_diff.append(diffevks)
        elif cond == 2:
            diffevks = mne.combine_evoked([evokeds['aligned'][pp], evokeds['early2_rdm'][pp]], 
                                                         weights=[-1, 1])
            early2rdm_diff.append(diffevks)
        elif cond == 3:
            diffevks = mne.combine_evoked([evokeds['aligned'][pp], evokeds['late1_rdm'][pp]], 
                                                         weights=[-1, 1])
            late1rdm_diff.append(diffevks)


In [15]:
#saved difference waves as data frame for FRN/ERN

difflists = ['early1rdm', 'late2rdm', 'early2rdm', 'late1rdm']

for diff in difflists:
    newdf = pd.DataFrame()
    for pp in participants:
        if diff == 'early1rdm':
            df = early1rdm_diff[pp].to_data_frame()
        elif diff == 'late2rdm':
            df = late2rdm_diff[pp].to_data_frame()
        elif diff == 'early2rdm':
            df = early2rdm_diff[pp].to_data_frame()
        elif diff == 'late1rdm':
            df = late1rdm_diff[pp].to_data_frame()
             
        time = df['time'].tolist()
        df = df[['FCz', 'F3', 'Fz', 'F4', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4']]
        ppdat = []
        for timept in range(0,800):
            subdf = df.loc[timept,:].mean()
            ppdat.append(subdf)
    
        newdf['pp%03d'% pp] = ppdat
    
    newdf['time'] = time
    newdf_filename = os.path.join('F:/Documents/Science/MirRevAdaptEEG/data/', 'Diffwaves_DF_%s_%s.csv' % (diff, erps))
    newdf.to_csv(newdf_filename)
        


In [16]:
#plot difference waves for FRN/ERN here in MNE
mne.viz.plot_compare_evokeds({'Early1rdm - Aligned':early1rdm_diff, 'Late2rdm - Aligned':late2rdm_diff, 'Early2rdm - Aligned':early2rdm_diff, 'Late1rdm - Aligned':late1rdm_diff}, 
                             picks=channels, show_sensors='upper right',
                             combine='mean',
                             title='Difference Wave');

combining channels using "mean"
combining channels using "mean"
combining channels using "mean"
combining channels using "mean"


In [17]:
#calculate difference waves for FRN/ERN - LATE MINUS EARLY
diffconds = ['rdm1_diff', 'rdm2_diff']
rdm1_diff = []
rdm2_diff = []


# the one below is actually the same output if we directly subtracted evokeds[early_rot] from evoked[late_rot], because
# the aligned condition subtracted in both conditions is the same
evokeds_diff = dict(early1rdm = early1rdm_diff,
               late2rdm = late2rdm_diff, 
               early2rdm = early2rdm_diff, 
               late1rdm = late1rdm_diff)

for cond in range(0, len(diffconds)):
    for pp in participants:
        if cond == 0:
            diffevks = mne.combine_evoked([evokeds_diff['early1rdm'][pp], evokeds_diff['late1rdm'][pp]], 
                                                            weights=[-1, 1])
            rdm1_diff.append(diffevks)
        elif cond == 1:
            diffevks = mne.combine_evoked([evokeds_diff['early2rdm'][pp], evokeds_diff['late2rdm'][pp]], 
                                                         weights=[-1, 1])
            rdm2_diff.append(diffevks)
        

In [18]:
#saved EARLY VS LATE difference waves as data frame for FRN/ERN

difflists = ['rdm1_diff', 'rdm2_diff']

for diff in difflists:
    newdf = pd.DataFrame()
    for pp in participants:
        if diff == 'rdm1_diff':
            df = rdm1_diff[pp].to_data_frame()
        elif diff == 'rdm2_diff':
            df = rdm2_diff[pp].to_data_frame()
             
        time = df['time'].tolist()
        df = df[['FCz', 'F3', 'Fz', 'F4', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4']]
        ppdat = []
        for timept in range(0,800):
            subdf = df.loc[timept,:].mean()
            ppdat.append(subdf)
    
        newdf['pp%03d'% pp] = ppdat
    
    newdf['time'] = time
    newdf_filename = os.path.join('F:/Documents/Science/MirRevAdaptEEG/data/', 'Diffwaves_DF_EvL_%s_%s.csv' % (diff, erps))
    newdf.to_csv(newdf_filename)

In [19]:
#plot EARLY VS LATE difference waves for FRN/ERN here in MNE
mne.viz.plot_compare_evokeds({'Rdm1':rdm1_diff, 'Rdm2':rdm2_diff}, 
                             picks=channels, show_sensors='upper right',
                             combine='mean',
                             title='Early Vs Late Difference Wave');

combining channels using "mean"
combining channels using "mean"
